# Join de pedidos, feriados e eventos de e-commerce — Last Mile SP ↔ SP

Este notebook parte do pressuposto de que a ETL anterior **já filtrou apenas pedidos em que vendedor e cliente estão no estado de São Paulo (SP ↔ SP)**.

O objetivo é enriquecer os pedidos com:

- feriados oficiais;
- distância até o feriado anterior e o próximo;
- janelas de 7 e 14 dias em torno de feriados;
- eventos comerciais relevantes para o e-commerce brasileiro;
- distância até o evento comercial anterior e o próximo;
- janelas de 7 e 14 dias em torno desses eventos;
- análise exploratória da taxa de atraso próxima aos eventos.

Como o escopo é **Last Mile**, todas as features temporais são relacionadas a `order_delivered_carrier_date`, utilizada no projeto como referência de entrada do pedido na etapa da transportadora.

> **Importante:** a taxa histórica de atraso por evento é calculada apenas para análise exploratória. Ela não é adicionada como feature ao dataset final, evitando target leakage.


In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


## 1. Inicialização do PySpark

Inicializa a sessão Spark com suporte ao acesso aos buckets S3 utilizados no projeto.


In [2]:
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import date

conf = SparkConf()
conf.set(
    "spark.jars.packages",
    "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.11.901"
)
conf.set(
    "spark.hadoop.fs.s3a.aws.credentials.provider",
    "com.amazonaws.auth.InstanceProfileCredentialsProvider"
)

spark = SparkSession.builder.config(conf=conf).getOrCreate()


:: loading settings :: url = jar:file:/usr/local/lib/python3.7/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2a462f81-670a-4582-85e8-d39e3e32136b;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 368ms :: artifacts dl 11ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 by [com.amazonaws#aws-java-sdk-bundle;1.12.262] in [default]
	---------------------------------------------------------------------
	|     

## 2. Leitura das tabelas

A tabela de pedidos já deve estar filtrada pela ETL para **SP ↔ SP**.

A tabela `tb_dates` contém os feriados utilizados no projeto.


In [3]:
tb_orders = spark.read.csv(
    "s3a://last-mile-optimization-trusted-sofhia/dataset-orders/final_table_orders_numeric.csv/",
    header=True,
    inferSchema=True
)

tb_orders_com_id = spark.read.csv(
    "s3a://last-mile-optimization-trusted-sofhia/dataset-orders/final_table_orders_numeric_com_order_id.csv/",
    header=True,
    inferSchema=True
)

tb_dates = spark.read.csv(
    "s3a://last-mile-optimization-trusted-sofhia/dataset-holidays/dates_2016_2018/",
    header=True,
    inferSchema=True
)

print("Quantidade de pedidos:", tb_orders.count())
print("Quantidade de registros de feriados:", tb_dates.count())

tb_dates.select("date", "localName").show(50, truncate=False)


26/11/08 02:11:10 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Quantidade de pedidos: 34856
Quantidade de registros de feriados: 42
+----------+------------------------------------+
|date      |localName                           |
+----------+------------------------------------+
|2016-01-01|Confraternização Universal          |
|2016-02-08|Carnaval                            |
|2016-02-09|Carnaval                            |
|2016-03-25|Sexta-feira Santa                   |
|2016-03-27|Domingo de Páscoa                   |
|2016-04-21|Dia de Tiradentes                   |
|2016-05-01|Dia do Trabalhador                  |
|2016-05-26|Corpus Christi                      |
|2016-07-09|Revolução Constitucionalista de 1932|
|2016-09-07|Dia da Independência                |
|2016-10-12|Nossa Senhora Aparecida             |
|2016-11-02|Dia de Finados                      |
|2016-11-15|Proclamação da República            |
|2016-12-25|Natal                               |
|2017-01-01|Confraternização Universal          |
|2017-02-27|Carnaval           

## 3. Preparação das datas

Os CSVs de feriados enviados possuem datas no formato `yyyy-MM-dd`, mas o código também aceita `dd/MM/yyyy`.

Para os pedidos, criamos `carrier_date_only` a partir de `order_delivered_carrier_date`. Essa será a data de referência das features porque o estudo está focado no **Last Mile**.


In [4]:
# Aceita tanto yyyy-MM-dd quanto dd/MM/yyyy
tb_dates_prep = tb_dates.withColumn(
    "data_feriado",
    F.coalesce(
        F.to_date(F.col("date"), "yyyy-MM-dd"),
        F.to_date(F.col("date"), "dd/MM/yyyy")
    )
)

tb_orders_prep = tb_orders_com_id.withColumn(
    "carrier_date_only",
    F.to_date(F.col("order_delivered_carrier_date"))
)

tb_dates_prep.select("date", "localName", "data_feriado").show(50, truncate=False)


+----------+------------------------------------+------------+
|date      |localName                           |data_feriado|
+----------+------------------------------------+------------+
|2016-01-01|Confraternização Universal          |2016-01-01  |
|2016-02-08|Carnaval                            |2016-02-08  |
|2016-02-09|Carnaval                            |2016-02-09  |
|2016-03-25|Sexta-feira Santa                   |2016-03-25  |
|2016-03-27|Domingo de Páscoa                   |2016-03-27  |
|2016-04-21|Dia de Tiradentes                   |2016-04-21  |
|2016-05-01|Dia do Trabalhador                  |2016-05-01  |
|2016-05-26|Corpus Christi                      |2016-05-26  |
|2016-07-09|Revolução Constitucionalista de 1932|2016-07-09  |
|2016-09-07|Dia da Independência                |2016-09-07  |
|2016-10-12|Nossa Senhora Aparecida             |2016-10-12  |
|2016-11-02|Dia de Finados                      |2016-11-02  |
|2016-11-15|Proclamação da República            |2016-1

## 4. Eventos comerciais relevantes para o e-commerce brasileiro

Feriados oficiais e eventos comerciais são tratados separadamente.

Além de datas como Natal, incluímos eventos que não são necessariamente feriados oficiais, mas podem estar associados a variações de demanda no e-commerce:

- Dia Internacional da Mulher (8 de março);
- Dia do Consumidor (15 de março);
- Dia das Mães (segundo domingo de maio);
- Dia dos Namorados (12 de junho);
- Dia dos Pais (segundo domingo de agosto);
- Dia do Cliente (15 de setembro);
- Dia das Crianças (12 de outubro);
- Halloween (31 de outubro);
- Black Friday (quarta sexta-feira de novembro);
- Cyber Monday (segunda-feira após a Black Friday);
- Natal (25 de dezembro).

Não atribuímos pesos manuais como “Black Friday = 3”. O modelo receberá características temporais objetivas e poderá aprender a relevância de cada período. A inclusão de um evento não implica, por si só, que ele cause atrasos.


In [5]:
ecommerce_events = [
    # 2016
    (date(2016, 3, 8), "Dia Internacional da Mulher"),
    (date(2016, 3, 15), "Dia do Consumidor"),
    (date(2016, 5, 8), "Dia das Maes"),
    (date(2016, 6, 12), "Dia dos Namorados"),
    (date(2016, 8, 14), "Dia dos Pais"),
    (date(2016, 9, 15), "Dia do Cliente"),
    (date(2016, 10, 12), "Dia das Criancas"),
    (date(2016, 10, 31), "Halloween"),
    (date(2016, 11, 25), "Black Friday"),
    (date(2016, 11, 28), "Cyber Monday"),
    (date(2016, 12, 25), "Natal"),

    # 2017
    (date(2017, 3, 8), "Dia Internacional da Mulher"),
    (date(2017, 3, 15), "Dia do Consumidor"),
    (date(2017, 5, 14), "Dia das Maes"),
    (date(2017, 6, 12), "Dia dos Namorados"),
    (date(2017, 8, 13), "Dia dos Pais"),
    (date(2017, 9, 15), "Dia do Cliente"),
    (date(2017, 10, 12), "Dia das Criancas"),
    (date(2017, 10, 31), "Halloween"),
    (date(2017, 11, 24), "Black Friday"),
    (date(2017, 11, 27), "Cyber Monday"),
    (date(2017, 12, 25), "Natal"),

    # 2018
    (date(2018, 3, 8), "Dia Internacional da Mulher"),
    (date(2018, 3, 15), "Dia do Consumidor"),
    (date(2018, 5, 13), "Dia das Maes"),
    (date(2018, 6, 12), "Dia dos Namorados"),
    (date(2018, 8, 12), "Dia dos Pais"),
    (date(2018, 9, 15), "Dia do Cliente"),
    (date(2018, 10, 12), "Dia das Criancas"),
    (date(2018, 10, 31), "Halloween"),
    (date(2018, 11, 23), "Black Friday"),
    (date(2018, 11, 26), "Cyber Monday"),
    (date(2018, 12, 25), "Natal")
]

tb_ecommerce_events = spark.createDataFrame(
    ecommerce_events,
    ["event_date", "event_name"]
)

tb_ecommerce_events.orderBy("event_date").show(50, truncate=False)


+----------+---------------------------+
|event_date|event_name                 |
+----------+---------------------------+
|2016-03-08|Dia Internacional da Mulher|
|2016-03-15|Dia do Consumidor          |
|2016-05-08|Dia das Maes               |
|2016-06-12|Dia dos Namorados          |
|2016-08-14|Dia dos Pais               |
|2016-09-15|Dia do Cliente             |
|2016-10-12|Dia das Criancas           |
|2016-10-31|Halloween                  |
|2016-11-25|Black Friday               |
|2016-11-28|Cyber Monday               |
|2016-12-25|Natal                      |
|2017-03-08|Dia Internacional da Mulher|
|2017-03-15|Dia do Consumidor          |
|2017-05-14|Dia das Maes               |
|2017-06-12|Dia dos Namorados          |
|2017-08-13|Dia dos Pais               |
|2017-09-15|Dia do Cliente             |
|2017-10-12|Dia das Criancas           |
|2017-10-31|Halloween                  |
|2017-11-24|Black Friday               |
|2017-11-27|Cyber Monday               |
|2017-12-25|Nata

## 5. Criação do calendário contínuo

Assim como no notebook original, criamos uma dimensão diária contínua. Isso evita fazer vários joins pesados diretamente entre pedidos, feriados e eventos.

O intervalo de 2015 a 2019 garante margem suficiente em torno dos dados de 2016–2018.


In [6]:
start_date = F.to_date(F.lit("2015-01-01"))
end_date = F.to_date(F.lit("2019-12-31"))

calendar_df = spark.range(1).select(
    F.explode(
        F.sequence(start_date, end_date, F.expr("interval 1 day"))
    ).alias("calendar_date")
)

calendar_df.show(5)


+-------------+
|calendar_date|
+-------------+
|   2015-01-01|
|   2015-01-02|
|   2015-01-03|
|   2015-01-04|
|   2015-01-05|
+-------------+
only showing top 5 rows



## 6. Features de feriados oficiais

Para cada dia do calendário são calculados:

- se o próprio dia é feriado;
- dias desde o último feriado;
- dias até o próximo feriado;
- presença de feriado nos próximos 7 e 14 dias;
- ocorrência de feriado nos últimos 7 e 14 dias.


In [7]:
calendar_with_holidays = calendar_df.join(
    tb_dates_prep.select("data_feriado").dropDuplicates(),
    calendar_df.calendar_date == tb_dates_prep.data_feriado,
    "left"
).withColumn(
    "is_holiday_today",
    F.when(F.col("data_feriado").isNotNull(), 1).otherwise(0)
).select(
    "calendar_date",
    "is_holiday_today"
)

window_past = Window.orderBy("calendar_date").rowsBetween(
    Window.unboundedPreceding, 0
)

window_future = Window.orderBy("calendar_date").rowsBetween(
    0, Window.unboundedFollowing
)

calendar_holiday_features = calendar_with_holidays.withColumn(
    "last_holiday_date",
    F.max(
        F.when(F.col("is_holiday_today") == 1, F.col("calendar_date"))
    ).over(window_past)
).withColumn(
    "next_holiday_date",
    F.min(
        F.when(F.col("is_holiday_today") == 1, F.col("calendar_date"))
    ).over(window_future)
)

calendar_holiday_features = calendar_holiday_features.withColumn(
    "days_since_last_holiday",
    F.coalesce(
        F.datediff(F.col("calendar_date"), F.col("last_holiday_date")),
        F.lit(999)
    )
).withColumn(
    "days_until_next_holiday",
    F.coalesce(
        F.datediff(F.col("next_holiday_date"), F.col("calendar_date")),
        F.lit(999)
    )
).withColumn(
    "is_holiday_in_7_days",
    F.when(F.col("days_until_next_holiday") <= 7, 1).otherwise(0)
).withColumn(
    "is_holiday_in_14_days",
    F.when(F.col("days_until_next_holiday") <= 14, 1).otherwise(0)
).withColumn(
    "had_holiday_7_days_ago",
    F.when(F.col("days_since_last_holiday") <= 7, 1).otherwise(0)
).withColumn(
    "had_holiday_14_days_ago",
    F.when(F.col("days_since_last_holiday") <= 14, 1).otherwise(0)
)

calendar_holiday_features.show(10)


26/11/08 02:11:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 0

+-------------+----------------+-----------------+-----------------+-----------------------+-----------------------+--------------------+---------------------+----------------------+-----------------------+
|calendar_date|is_holiday_today|last_holiday_date|next_holiday_date|days_since_last_holiday|days_until_next_holiday|is_holiday_in_7_days|is_holiday_in_14_days|had_holiday_7_days_ago|had_holiday_14_days_ago|
+-------------+----------------+-----------------+-----------------+-----------------------+-----------------------+--------------------+---------------------+----------------------+-----------------------+
|   2015-01-01|               0|             null|       2016-01-01|                    999|                    365|                   0|                    0|                     0|                      0|
|   2015-01-02|               0|             null|       2016-01-01|                    999|                    364|                   0|                    0|             

## 7. Features dos eventos de e-commerce

Aplicamos a mesma lógica aos eventos comerciais.

As novas features serão:

- `ecommerce_event_at_carrier`;
- `previous_ecommerce_event`;
- `next_ecommerce_event`;
- `days_since_last_ecommerce_event`;
- `days_until_next_ecommerce_event`;
- `is_ecommerce_event_in_7_days`;
- `is_ecommerce_event_in_14_days`;
- `had_ecommerce_event_7_days_ago`;
- `had_ecommerce_event_14_days_ago`.

Os nomes do evento anterior/próximo são mantidos para análise. Antes de treinar modelos que exigem somente números, essas colunas podem ser removidas ou codificadas.


In [8]:
calendar_with_events = calendar_df.join(
    tb_ecommerce_events,
    calendar_df.calendar_date == tb_ecommerce_events.event_date,
    "left"
).withColumn(
    "is_ecommerce_event_today",
    F.when(F.col("event_date").isNotNull(), 1).otherwise(0)
)

calendar_event_features = calendar_with_events.withColumn(
    "last_ecommerce_event_date",
    F.max(
        F.when(
            F.col("is_ecommerce_event_today") == 1,
            F.col("calendar_date")
        )
    ).over(window_past)
).withColumn(
    "next_ecommerce_event_date",
    F.min(
        F.when(
            F.col("is_ecommerce_event_today") == 1,
            F.col("calendar_date")
        )
    ).over(window_future)
)

# Mapeamento data -> nome para recuperar os nomes do evento anterior/próximo
event_lookup_previous = tb_ecommerce_events.select(
    F.col("event_date").alias("previous_event_date_lookup"),
    F.col("event_name").alias("previous_ecommerce_event")
)

event_lookup_next = tb_ecommerce_events.select(
    F.col("event_date").alias("next_event_date_lookup"),
    F.col("event_name").alias("next_ecommerce_event")
)

calendar_event_features = calendar_event_features.join(
    event_lookup_previous,
    calendar_event_features.last_ecommerce_event_date ==
    event_lookup_previous.previous_event_date_lookup,
    "left"
).join(
    event_lookup_next,
    calendar_event_features.next_ecommerce_event_date ==
    event_lookup_next.next_event_date_lookup,
    "left"
)

calendar_event_features = calendar_event_features.withColumn(
    "days_since_last_ecommerce_event",
    F.coalesce(
        F.datediff(
            F.col("calendar_date"),
            F.col("last_ecommerce_event_date")
        ),
        F.lit(999)
    )
).withColumn(
    "days_until_next_ecommerce_event",
    F.coalesce(
        F.datediff(
            F.col("next_ecommerce_event_date"),
            F.col("calendar_date")
        ),
        F.lit(999)
    )
).withColumn(
    "is_ecommerce_event_in_7_days",
    F.when(F.col("days_until_next_ecommerce_event") <= 7, 1).otherwise(0)
).withColumn(
    "is_ecommerce_event_in_14_days",
    F.when(F.col("days_until_next_ecommerce_event") <= 14, 1).otherwise(0)
).withColumn(
    "had_ecommerce_event_7_days_ago",
    F.when(F.col("days_since_last_ecommerce_event") <= 7, 1).otherwise(0)
).withColumn(
    "had_ecommerce_event_14_days_ago",
    F.when(F.col("days_since_last_ecommerce_event") <= 14, 1).otherwise(0)
)

calendar_event_features.select(
    "calendar_date",
    "is_ecommerce_event_today",
    "previous_ecommerce_event",
    "next_ecommerce_event",
    "days_since_last_ecommerce_event",
    "days_until_next_ecommerce_event",
    "is_ecommerce_event_in_7_days",
    "is_ecommerce_event_in_14_days",
    "had_ecommerce_event_7_days_ago",
    "had_ecommerce_event_14_days_ago"
).filter(
    (F.col("days_since_last_ecommerce_event") <= 3) |
    (F.col("days_until_next_ecommerce_event") <= 3)
).show(50, truncate=False)


26/11/08 02:11:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 0

+-------------+------------------------+---------------------------+---------------------------+-------------------------------+-------------------------------+----------------------------+-----------------------------+------------------------------+-------------------------------+
|calendar_date|is_ecommerce_event_today|previous_ecommerce_event   |next_ecommerce_event       |days_since_last_ecommerce_event|days_until_next_ecommerce_event|is_ecommerce_event_in_7_days|is_ecommerce_event_in_14_days|had_ecommerce_event_7_days_ago|had_ecommerce_event_14_days_ago|
+-------------+------------------------+---------------------------+---------------------------+-------------------------------+-------------------------------+----------------------------+-----------------------------+------------------------------+-------------------------------+
|2016-03-05   |0                       |null                       |Dia Internacional da Mulher|999                            |3                      

## 8. União das features em uma única dimensão de calendário

Aqui juntamos as features de feriados e de eventos comerciais. Depois será necessário apenas um join dessa dimensão com os pedidos.


In [9]:
calendar_final_features = calendar_holiday_features.select(
    "calendar_date",
    "is_holiday_today",
    "days_since_last_holiday",
    "days_until_next_holiday",
    "is_holiday_in_7_days",
    "is_holiday_in_14_days",
    "had_holiday_7_days_ago",
    "had_holiday_14_days_ago"
).join(
    calendar_event_features.select(
        "calendar_date",
        "is_ecommerce_event_today",
        "previous_ecommerce_event",
        "next_ecommerce_event",
        "days_since_last_ecommerce_event",
        "days_until_next_ecommerce_event",
        "is_ecommerce_event_in_7_days",
        "is_ecommerce_event_in_14_days",
        "had_ecommerce_event_7_days_ago",
        "had_ecommerce_event_14_days_ago"
    ),
    "calendar_date",
    "left"
).withColumnRenamed(
    "calendar_date",
    "match_date"
)

calendar_final_features.show(10, truncate=False)


26/11/08 02:11:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 0

+----------+----------------+-----------------------+-----------------------+--------------------+---------------------+----------------------+-----------------------+------------------------+------------------------+---------------------------+-------------------------------+-------------------------------+----------------------------+-----------------------------+------------------------------+-------------------------------+
|match_date|is_holiday_today|days_since_last_holiday|days_until_next_holiday|is_holiday_in_7_days|is_holiday_in_14_days|had_holiday_7_days_ago|had_holiday_14_days_ago|is_ecommerce_event_today|previous_ecommerce_event|next_ecommerce_event       |days_since_last_ecommerce_event|days_until_next_ecommerce_event|is_ecommerce_event_in_7_days|is_ecommerce_event_in_14_days|had_ecommerce_event_7_days_ago|had_ecommerce_event_14_days_ago|
+----------+----------------+-----------------------+-----------------------+--------------------+---------------------+----------------

## 9. Join com os pedidos

O join é realizado entre `carrier_date_only` e o calendário enriquecido.

Dessa forma, as features representam o contexto temporal existente quando o pedido entra na etapa da transportadora.

A coluna `carrier_month` representa o mês (1 a 12) da chegada do pedido à transportadora, sendo calculada antes da seleção das colunas finais e preservada nas duas exportações.


In [10]:
df_final = tb_orders_prep.join(
    calendar_final_features,
    tb_orders_prep.carrier_date_only == calendar_final_features.match_date,
    "left"
).withColumn(
    "holiday_at_carrier",
    F.coalesce(F.col("is_holiday_today"), F.lit(0))
).withColumn(
    "ecommerce_event_at_carrier",
    F.coalesce(F.col("is_ecommerce_event_today"), F.lit(0))
)

df_final = df_final.withColumn(
    "carrier_month",
    F.month(F.to_timestamp(F.col("order_delivered_carrier_date")))
)

df_final = df_final.select(
    "order_id",
    # Datas originais
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "interval_code_delivered_carrier",
    "carrier_month",
    "order_delivered_customer_date",
    "interval_code_delivered_customer",
    "order_estimated_delivery_date",
    "shipping_limit_date",

    # Features de feriados
    "holiday_at_carrier",
    "days_since_last_holiday",
    "days_until_next_holiday",
    "is_holiday_in_7_days",
    "is_holiday_in_14_days",
    "had_holiday_7_days_ago",
    "had_holiday_14_days_ago",

    # Features de eventos de e-commerce
    "ecommerce_event_at_carrier",
    "previous_ecommerce_event",
    "next_ecommerce_event",
    "days_since_last_ecommerce_event",
    "days_until_next_ecommerce_event",
    "is_ecommerce_event_in_7_days",
    "is_ecommerce_event_in_14_days",
    "had_ecommerce_event_7_days_ago",
    "had_ecommerce_event_14_days_ago",

    # Demais features já existentes
    "purchase_hour",
    "purchase_day_of_week",
    "is_weekend",
    "delivered_on_time",
    "delivery_time_days",
    "price",
    "freight_value",
    "product_weight_g",
    "volume_cm3",
    "distance_km",
    "category_name",
    "customer_city",
    "seller_city",
    "same_city",
    "review_score"
).dropna(
    subset=["product_weight_g", "volume_cm3"]
)

# Remove pesos zerados e negativos antes de gerar as duas versões.
df_final = df_final.filter(F.col("product_weight_g") > 0)
df_final.show(10, truncate=False)


26/11/08 02:11:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 0

+--------------------------------+------------------------+-------------------+----------------------------+-------------------------------+-------------+-----------------------------+--------------------------------+-----------------------------+-------------------+------------------+-----------------------+-----------------------+--------------------+---------------------+----------------------+-----------------------+--------------------------+------------------------+---------------------------+-------------------------------+-------------------------------+----------------------------+-----------------------------+------------------------------+-------------------------------+-------------+--------------------+----------+-----------------+------------------+------+-------------+----------------+----------+-----------+---------------+-----------------------+---------------------+---------+------------+
|order_id                        |order_purchase_timestamp|order_approved_at  |o

## 9.1. Pedidos já atrasados na chegada à transportadora

Calculamos os dias restantes entre a chegada à transportadora e a data estimada de entrega. Pedidos com resultado **negativo** já haviam ultrapassado o prazo previsto ao iniciar o Last Mile e são excluídos da tabela final. O diagnóstico abaixo mostra quantos pedidos se enquadram nessa situação, antes da exclusão.

A comparação utiliza datas (sem horários), seguindo a regra de dias apresentada. Valores nulos nas datas são contabilizados separadamente e preservados para avaliação posterior.

In [11]:
# Dias disponíveis para a transportadora realizar a entrega
# datediff é a alternativa PySpark ao .dt.days do pandas.
df_final = df_final.withColumn(
    "carrier_available_days",
    F.datediff(
        F.to_date(F.col("order_estimated_delivery_date")),
        F.to_date(F.col("order_delivered_carrier_date"))
    )
)

# Diagnóstico antes de remover pedidos
resumo_prazo = df_final.agg(
    F.count("*").alias("total_pedidos"),
    F.sum(F.when(F.col("carrier_available_days") < 0, 1).otherwise(0)).alias("ja_atrasados_na_transportadora"),
    F.sum(F.when(F.col("carrier_available_days") >= 0, 1).otherwise(0)).alias("dentro_do_prazo_na_transportadora"),
    F.sum(F.when(F.col("carrier_available_days").isNull(), 1).otherwise(0)).alias("sem_prazo_calculavel")
).withColumn(
    "percentual_ja_atrasados",
    F.round(F.col("ja_atrasados_na_transportadora") * 100.0 / F.col("total_pedidos"), 2)
)

resumo_prazo.show(truncate=False)

# Exibir alguns exemplos de pedidos que serão excluídos
df_final.filter(F.col("carrier_available_days") < 0).select(
    "order_delivered_carrier_date", "order_estimated_delivery_date", "carrier_available_days"
).show(10, truncate=False)

# Excluir somente os pedidos cujo prazo é negativo; manter nulos para tratamento separado
df_final = df_final.filter(
    (F.col("carrier_available_days") >= 0) |
    F.col("carrier_available_days").isNull()
)

print(f"Pedidos restantes na tabela final: {df_final.count()}")
print("Pedidos com prazo negativo após o filtro:", df_final.filter(F.col("carrier_available_days") < 0).count())

26/11/08 02:11:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 0

+-------------+------------------------------+---------------------------------+--------------------+-----------------------+
|total_pedidos|ja_atrasados_na_transportadora|dentro_do_prazo_na_transportadora|sem_prazo_calculavel|percentual_ja_atrasados|
+-------------+------------------------------+---------------------------------+--------------------+-----------------------+
|34852        |226                           |34626                            |0                   |0.65                   |
+-------------+------------------------------+---------------------------------+--------------------+-----------------------+



26/11/08 02:11:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 0

+----------------------------+-----------------------------+----------------------+
|order_delivered_carrier_date|order_estimated_delivery_date|carrier_available_days|
+----------------------------+-----------------------------+----------------------+
|2017-03-30 16:39:56         |2017-03-16 00:00:00          |-14                   |
|2017-12-14 21:35:19         |2017-12-12 00:00:00          |-2                    |
|2017-12-22 00:47:04         |2017-12-21 00:00:00          |-1                    |
|2017-12-22 23:12:37         |2017-12-19 00:00:00          |-3                    |
|2018-01-31 01:27:46         |2018-01-12 00:00:00          |-19                   |
|2018-02-07 15:38:51         |2018-02-01 00:00:00          |-6                    |
|2018-03-02 00:04:40         |2018-02-26 00:00:00          |-4                    |
|2018-04-26 15:10:00         |2018-04-17 00:00:00          |-9                    |
|2018-04-27 17:50:00         |2018-03-28 00:00:00          |-30             

26/11/08 02:11:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:11:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 0

Pedidos restantes na tabela final: 34626


26/11/08 02:12:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 0

Pedidos com prazo negativo após o filtro: 0


26/11/08 02:12:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## 9.2. Expurgo temporal do hold-out de otimização (01/05/2018)

Para reproduzir a regra do notebook `XGBoost_oversampling_v3.ipynb`, removemos os pedidos que **entraram na transportadora antes de 01/05/2018** e **foram entregues ao cliente em 01/05/2018 ou depois**. No corte de validação, o desfecho desses pedidos ainda não estaria disponível para o treino de tuning.

O filtro é aplicado **antes das análises e das duas exportações**, de modo que os arquivos com e sem `order_id` usem a mesma população. As comparações são feitas por timestamp, com o corte em `2018-05-01 00:00:00`. A quantidade removida é apurada em execução: **343 é o número observado no XGBoost anterior**, e pode diferir se a base ou os filtros anteriores mudaram.

**Atenção metodológica:** este expurgo global é uma escolha de preparação da base. Ele não substitui a verificação independente de sobreposições no corte de teste de 01/07/2018.


In [12]:
# Regra equivalente ao expurgo do hold-out de tuning no XGBoost
from pyspark.sql import functions as F

data_corte_validacao = F.to_timestamp(F.lit("2018-05-01 00:00:00"))

condicao_expurgo_maio = (
    (F.to_timestamp(F.col("order_delivered_carrier_date")) < data_corte_validacao)
    & (F.to_timestamp(F.col("order_delivered_customer_date")) >= data_corte_validacao)
)

total_antes_expurgo = df_final.count()
pedidos_expurgados_maio = df_final.filter(condicao_expurgo_maio)
qtd_expurgados_maio = pedidos_expurgados_maio.count()

print(f"Registros antes do expurgo: {total_antes_expurgo:,}")
print(f"Registros expurgados (corte 01/05/2018): {qtd_expurgados_maio:,}")
print("Exemplos dos registros excluídos:")
pedidos_expurgados_maio.select(
    "order_id", "order_delivered_carrier_date", "order_delivered_customer_date"
).show(10, truncate=False)

# Preserva registros com datas nulas, tal como a seleção negativa da condição.
df_final = df_final.filter(~F.coalesce(condicao_expurgo_maio, F.lit(False)))

print(f"Registros restantes: {df_final.count():,}")
print("Registros que ainda satisfazem o expurgo:", df_final.filter(condicao_expurgo_maio).count())
assert df_final.count() == total_antes_expurgo - qtd_expurgados_maio


26/11/08 02:12:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 0

Registros antes do expurgo: 34,626
Registros expurgados (corte 01/05/2018): 343
Exemplos dos registros excluídos:


26/11/08 02:12:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 0

+--------------------------------+----------------------------+-----------------------------+
|order_id                        |order_delivered_carrier_date|order_delivered_customer_date|
+--------------------------------+----------------------------+-----------------------------+
|09ae30c3aa82707a515bda0df370f7d2|2018-04-16 23:21:51         |2018-05-03 17:05:11          |
|0e4947343ae2d8c37f2fc453e057c567|2018-04-20 21:24:58         |2018-05-07 17:38:31          |
|002175704e8b209f61b9ad5cfd92b60e|2018-04-23 19:03:19         |2018-05-02 20:38:44          |
|00c826c901ab01b6f598a8f0c9b56b82|2018-04-24 15:55:40         |2018-05-02 16:28:42          |
|72d78d466fc73ada5f9cc332c9beeedc|2018-04-25 13:58:00         |2018-05-05 14:38:30          |
|72d78d466fc73ada5f9cc332c9beeedc|2018-04-25 13:58:00         |2018-05-05 14:38:30          |
|7b1e7ecf29040bd9581a1577a522548d|2018-04-25 16:07:00         |2018-05-02 21:13:00          |
|7b1e7ecf29040bd9581a1577a522548d|2018-04-25 16:07:00       

26/11/08 02:12:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:09 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 0

Registros restantes: 34,283


26/11/08 02:12:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 0

Registros que ainda satisfazem o expurgo: 0


26/11/08 02:12:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 0

In [13]:
df_final = (
    df_final
    .withColumn(
        "delivery_time_per_km",
        F.round(
            F.col("carrier_available_days") /
            F.col("distance_km"),
            6
        )
    )
    .withColumn(
        "freight_per_kg",
        F.round(
            F.col("freight_value") /
            (F.col("product_weight_g") / 1000),
            6
        )
    )
    .withColumn(
        "product_density",
        F.round(
            F.col("product_weight_g") /
            F.col("volume_cm3"),
            6
        )
    )
)


## 10. Validação das novas features

As células abaixo permitem conferir quantos pedidos entraram no last mile exatamente em feriados ou eventos comerciais.


In [14]:
print("Pedidos por holiday_at_carrier:")
df_final.groupBy("holiday_at_carrier").count().show()

print("Pedidos por ecommerce_event_at_carrier:")
df_final.groupBy("ecommerce_event_at_carrier").count().show()


Pedidos por holiday_at_carrier:


26/11/08 02:12:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 0

+------------------+-----+
|holiday_at_carrier|count|
+------------------+-----+
|                 1|   32|
|                 0|34251|
+------------------+-----+

Pedidos por ecommerce_event_at_carrier:


26/11/08 02:12:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:18 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 0

+--------------------------+-----+
|ecommerce_event_at_carrier|count|
+--------------------------+-----+
|                         1|  905|
|                         0|33378|
+--------------------------+-----+



In [15]:
# Pedidos cuja entrada no last mile ocorreu exatamente em um evento comercial
df_final.filter(
    F.col("ecommerce_event_at_carrier") == 1
).select(
    "order_delivered_carrier_date",
    "previous_ecommerce_event",
    "next_ecommerce_event",
    "delivered_on_time"
).show(50, truncate=False)


26/11/08 02:12:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 0

+----------------------------+---------------------------+---------------------------+-----------------+
|order_delivered_carrier_date|previous_ecommerce_event   |next_ecommerce_event       |delivered_on_time|
+----------------------------+---------------------------+---------------------------+-----------------+
|2016-10-12 20:38:26         |Dia das Criancas           |Dia das Criancas           |1                |
|2016-10-12 03:51:25         |Dia das Criancas           |Dia das Criancas           |1                |
|2016-10-12 16:34:44         |Dia das Criancas           |Dia das Criancas           |1                |
|2016-10-12 18:34:07         |Dia das Criancas           |Dia das Criancas           |1                |
|2017-03-08 08:24:38         |Dia Internacional da Mulher|Dia Internacional da Mulher|1                |
|2017-03-08 01:17:48         |Dia Internacional da Mulher|Dia Internacional da Mulher|1                |
|2017-03-08 12:41:29         |Dia Internacional da Mulh

26/11/08 02:12:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## 11. Análise exploratória: atraso próximo aos eventos

Esta etapa responde à pergunta:

**Pedidos que entram no last mile próximos de Black Friday, Natal, Dia das Mães etc. apresentam maior taxa de atraso?**

Usaremos inicialmente uma janela de:

- até **14 dias antes** do evento;
- até **7 dias depois** do evento.

A variável `delivered_on_time` é usada apenas para medir o resultado histórico.

> **Esta taxa não será incorporada como feature no dataset final.** Fazer isso usando toda a base poderia introduzir target leakage.


In [16]:
# Para esta análise exploratória, cruzamos somente as datas necessárias.
orders_events_analysis = tb_orders_prep.select(
    "order_delivered_carrier_date",
    "carrier_date_only",
    "delivered_on_time"
).crossJoin(
    F.broadcast(tb_ecommerce_events)
).withColumn(
    "days_from_event",
    F.datediff(
        F.col("carrier_date_only"),
        F.col("event_date")
    )
)

orders_near_events = orders_events_analysis.filter(
    (F.col("days_from_event") >= -14) &
    (F.col("days_from_event") <= 7)
)

orders_near_events.select(
    "carrier_date_only",
    "event_date",
    "event_name",
    "days_from_event",
    "delivered_on_time"
).show(20, truncate=False)


+-----------------+----------+-----------------+---------------+-----------------+
|carrier_date_only|event_date|event_name       |days_from_event|delivered_on_time|
+-----------------+----------+-----------------+---------------+-----------------+
|2017-05-04       |2017-05-14|Dia das Maes     |-10            |1                |
|2018-08-10       |2018-08-12|Dia dos Pais     |-2             |1                |
|2017-12-12       |2017-12-25|Natal            |-13            |1                |
|2018-07-30       |2018-08-12|Dia dos Pais     |-13            |0                |
|2018-05-16       |2018-05-13|Dia das Maes     |3              |1                |
|2017-08-15       |2017-08-13|Dia dos Pais     |2              |1                |
|2017-11-22       |2017-11-24|Black Friday     |-2             |0                |
|2017-11-22       |2017-11-27|Cyber Monday     |-5             |0                |
|2017-11-27       |2017-11-24|Black Friday     |3              |0                |
|201

### Taxa de atraso por evento

Considerando a codificação utilizada no projeto:

- `delivered_on_time = 1` → entrega no prazo;
- `delivered_on_time = 0` → entrega atrasada.

A tabela abaixo calcula a quantidade de pedidos e a taxa de atraso observada em torno de cada evento.


In [17]:
event_analysis = orders_near_events.groupBy(
    "event_name"
).agg(
    F.count("*").alias("total_orders"),
    F.sum(
        F.when(F.col("delivered_on_time") == 0, 1).otherwise(0)
    ).alias("delayed_orders")
).withColumn(
    "delay_rate",
    F.round(
        F.col("delayed_orders") / F.col("total_orders"),
        4
    )
).withColumn(
    "delay_rate_percent",
    F.round(F.col("delay_rate") * 100, 2)
)

event_analysis.orderBy(
    F.desc("delay_rate")
).show(truncate=False)


+---------------------------+------------+--------------+----------+------------------+
|event_name                 |total_orders|delayed_orders|delay_rate|delay_rate_percent|
+---------------------------+------------+--------------+----------+------------------+
|Dia dos Pais               |3233        |455           |0.1407    |14.07             |
|Dia do Consumidor          |2443        |278           |0.1138    |11.38             |
|Dia Internacional da Mulher|2222        |248           |0.1116    |11.16             |
|Black Friday               |1973        |139           |0.0705    |7.05              |
|Cyber Monday               |2035        |139           |0.0683    |6.83              |
|Natal                      |1341        |67            |0.05      |5.0               |
|Dia das Maes               |2842        |135           |0.0475    |4.75              |
|Halloween                  |1168        |53            |0.0454    |4.54              |
|Dia do Cliente             |974

### Separação entre período anterior e posterior ao evento

Para Last Mile, essa comparação pode ser útil porque a pressão logística pode se manifestar de maneira diferente antes e depois de uma data comercial.

- `PRE_EVENT` → pedido entrou no last mile entre 14 e 1 dia antes;
- `EVENT_DAY` → entrou exatamente na data do evento;
- `POST_EVENT` → entrou entre 1 e 7 dias depois.


In [18]:
orders_near_events_period = orders_near_events.withColumn(
    "event_period",
    F.when(F.col("days_from_event") < 0, "PRE_EVENT")
     .when(F.col("days_from_event") == 0, "EVENT_DAY")
     .otherwise("POST_EVENT")
)

event_period_analysis = orders_near_events_period.groupBy(
    "event_name",
    "event_period"
).agg(
    F.count("*").alias("total_orders"),
    F.sum(
        F.when(F.col("delivered_on_time") == 0, 1).otherwise(0)
    ).alias("delayed_orders")
).withColumn(
    "delay_rate_percent",
    F.round(
        (F.col("delayed_orders") / F.col("total_orders")) * 100,
        2
    )
)

event_period_analysis.orderBy(
    "event_name",
    "event_period"
).show(100, truncate=False)


+---------------------------+------------+------------+--------------+------------------+
|event_name                 |event_period|total_orders|delayed_orders|delay_rate_percent|
+---------------------------+------------+------------+--------------+------------------+
|Black Friday               |EVENT_DAY   |137         |7             |5.11              |
|Black Friday               |POST_EVENT  |963         |86            |8.93              |
|Black Friday               |PRE_EVENT   |873         |46            |5.27              |
|Cyber Monday               |EVENT_DAY   |199         |6             |3.02              |
|Cyber Monday               |POST_EVENT  |887         |81            |9.13              |
|Cyber Monday               |PRE_EVENT   |949         |52            |5.48              |
|Dia Internacional da Mulher|EVENT_DAY   |138         |20            |14.49             |
|Dia Internacional da Mulher|POST_EVENT  |684         |74            |10.82             |
|Dia Inter

## 12. Comparação com a taxa geral de atraso

A taxa geral serve como referência para interpretar os eventos.

Por exemplo, se a taxa geral de atraso for 8% e a janela da Black Friday apresentar 15%, existe um sinal descritivo de maior concentração de atrasos naquele período. Isso **não prova causalidade**, mas pode justificar investigar a variável no modelo.


In [19]:
overall_delay = df_final.agg(
    F.count("*").alias("total_orders"),
    F.sum(
        F.when(F.col("delivered_on_time") == 0, 1).otherwise(0)
    ).alias("delayed_orders")
).withColumn(
    "overall_delay_rate_percent",
    F.round(
        (F.col("delayed_orders") / F.col("total_orders")) * 100,
        2
    )
)

overall_delay.show(truncate=False)


26/11/08 02:12:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 0

+------------+--------------+--------------------------+
|total_orders|delayed_orders|overall_delay_rate_percent|
+------------+--------------+--------------------------+
|34283       |1888          |5.51                      |
+------------+--------------+--------------------------+



## 13. Verificação de valores nulos

Confere se alguma das features ficou sem valor após os joins.


In [20]:
null_counts = [
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df_final.columns
]

df_final.select(null_counts).show(truncate=False)


26/11/08 02:12:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 0

+--------+------------------------+-----------------+----------------------------+-------------------------------+-------------+-----------------------------+--------------------------------+-----------------------------+-------------------+------------------+-----------------------+-----------------------+--------------------+---------------------+----------------------+-----------------------+--------------------------+------------------------+--------------------+-------------------------------+-------------------------------+----------------------------+-----------------------------+------------------------------+-------------------------------+-------------+--------------------+----------+-----------------+------------------+-----+-------------+----------------+----------+-----------+-------------+-------------+-----------+---------+------------+----------------------+--------------------+--------------+---------------+
|order_id|order_purchase_timestamp|order_approved_at|order_de

## 14. Dataset numérico para Machine Learning

`previous_ecommerce_event` e `next_ecommerce_event` são colunas categóricas em texto.

Para um treinamento que utilizará somente as variáveis numéricas, podemos removê-las. As demais features temporais dos eventos continuam disponíveis para o XGBoost.

Se posteriormente houver interesse em avaliar o nome do evento como variável, ele deve ser codificado adequadamente dentro do pipeline de treinamento.


In [21]:
df_final_ml = df_final.drop(
    "previous_ecommerce_event",
    "next_ecommerce_event"
)

df_final_ml_com_id = df_final_ml
# Mantém o arquivo de ML com o mesmo esquema anterior.
df_final_ml = df_final_ml.drop("order_id")

print("Colunas disponíveis para a próxima etapa:")
for c in df_final_ml.columns:
    print(c)


Colunas disponíveis para a próxima etapa:
order_purchase_timestamp
order_approved_at
order_delivered_carrier_date
interval_code_delivered_carrier
carrier_month
order_delivered_customer_date
interval_code_delivered_customer
order_estimated_delivery_date
shipping_limit_date
holiday_at_carrier
days_since_last_holiday
days_until_next_holiday
is_holiday_in_7_days
is_holiday_in_14_days
had_holiday_7_days_ago
had_holiday_14_days_ago
ecommerce_event_at_carrier
days_since_last_ecommerce_event
days_until_next_ecommerce_event
is_ecommerce_event_in_7_days
is_ecommerce_event_in_14_days
had_ecommerce_event_7_days_ago
had_ecommerce_event_14_days_ago
purchase_hour
purchase_day_of_week
is_weekend
delivered_on_time
delivery_time_days
price
freight_value
product_weight_g
volume_cm3
distance_km
category_name
customer_city
seller_city
same_city
review_score
carrier_available_days
delivery_time_per_km
freight_per_kg
product_density


## 15. Exportação para a camada Client

O arquivo final contém as features originais de feriados e as novas features relacionadas aos eventos comerciais.

A análise de `delay_rate` não é exportada como feature.


A exportação utiliza `df_final_ml`, derivado de `df_final` já filtrado para excluir prazos negativos na chegada à transportadora.
As duas exportações também aplicam o expurgo temporal de 01/05/2018 da seção 9.2.


In [22]:
df_final_ml.coalesce(1) \
    .write \
    .option("header", "true") \
    .mode("overwrite") \
    .csv(
        "s3a://last-mile-optimization-client-sofhia/table_orders_holidays_ecommerce_events.csv"
    )

print("Dataset exportado com sucesso.")

# Segunda tabela: mesmo conteúdo, com order_id para análises.
df_final_ml_com_id.coalesce(1) \
    .write.option("header", "true").mode("overwrite") \
    .csv("s3a://last-mile-optimization-client-sofhia/table_orders_holidays_ecommerce_events_com_order_id.csv")

print("Linhas finais:", df_final_ml_com_id.count())
print("Pedidos únicos:", df_final_ml_com_id.select("order_id").distinct().count())


26/11/08 02:12:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 0

Dataset exportado com sucesso.


26/11/08 02:12:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 02:12:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/08 0

Linhas finais: 34283


Pedidos únicos: 29839


## 16. Encerramento

Ao finalizar todas as operações, encerramos a sessão Spark.


In [23]:
spark.stop()
